## **Random Forest STL na embeddingach MoLFormer**

Wykorzystana reprezentacja: **MoLFormer embeddings**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. Half Life (Obach)
6. Clearance Hepatocyte (AstraZeneca)
7. CYP3A4 Inhibition (Veith)
8. VDss (Lombardo)
9. AMES Mutagenicity

Metryki jakości: AUROC (klasyfikacja), RMSE(regresja), Accuracy, F1
(klasyfikacja), R^2(regresja), MAE(regresja)

In [1]:
#UWAGA: jak pobierzesz dane dla danego endpointu to sprwdź ilość NaN i w zależności od ich ilości będziemy je usuwać albo zastępować średnią/medianą albo robić jedno i drugie i sprawdzać co da lepszy efekt, na razie możesz je po prostu usuwać

In [2]:
#użyj parametru class_weight='balanced' przez różnice w ilości elementów w klasach

In [3]:
#trzeba dodać zapisywanie danych testowych i treniengowych do plików żeby później wczytać dokładnie te same do MTL żeby wyniki porównania były miarodajne, bo jak nam się wylosują różne zbiory to będzie trochę słabo
#nadawaj im prosze jakieś jasne nazwy żeby łatwo to było potem ogarnąć

Krok 1: Przygotowanie danych — wczytanie embeddingów z plików CSV

In [12]:
import numpy as np
import pandas as pd
import os

from google.colab import drive
drive.mount('/content/drive')

# embeddings_folder = '/content/drive/MyDrive/MLDD - ADMET/STL_ML'
embeddings_folder = '/content/drive/MyDrive/mldd_data'


def load_embeddings(dataset_name, train_df, test_df):
    """Wczytuje CSV z embeddingami i dzieli na train/test po SMILES (tak samo jak featurizer)."""
    file_path = os.path.join(embeddings_folder, f'{dataset_name}_MoLFormer_embeddings.csv')
    emb_df = pd.read_csv(file_path)

    train_smiles = set(train_df['Drug'].tolist())
    test_smiles  = set(test_df['Drug'].tolist())

    emb_train = emb_df[emb_df['Drug'].isin(train_smiles)]
    emb_test  = emb_df[emb_df['Drug'].isin(test_smiles)]

    feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]

    X_train = emb_train[feature_cols].values
    y_train = emb_train['Y'].values
    X_test  = emb_test[feature_cols].values
    y_test  = emb_test['Y'].values

    return X_train, y_train, X_test, y_test

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pickle

def split_and_save(tdc_data, dataset_name):
    split    = tdc_data.get_split(seed=42, frac=[0.8, 0.0, 0.2])
    train_df = split["train"]
    test_df  = split["test"]

    filepath = f"{dataset_name}_split.pkl"
    with open(filepath, "wb") as f:
        pickle.dump({"train": train_df, "test": test_df}, f)

    return train_df, test_df

In [6]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [7]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    print(f"Best params: {metrics['best_params']}")
    print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        f.write(f"Best params: {metrics['best_params']}\n")
        f.write(f"{'-'*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

Krok 2: Trening Random Forest

In [8]:
#kod random forest regression
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_rf_regression(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
            "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]

        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred = final_model.predict(X_test)

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
            "mae":  mean_absolute_error(y_test, y_test_pred),
            "r2":   r2_score(y_test, y_test_pred)
        },
        "model": final_model
    }

    return metrics

In [9]:
#kod random forest klasyfikacja
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def train_rf_classification(X_train, X_test, y_train, y_test, rf_param_grid=None, cv_folds=5):

    if rf_param_grid is None:
        rf_param_grid = {
          "n_estimators": [200, 500, 1000, 1500],
            "max_depth": [None, 10,15,20,25,30],
            "min_samples_split": [2, 5,10],
            "max_features": ["sqrt", "log2"]
        }

    n_iter = 1
    for values in rf_param_grid.values():
        n_iter *= len(values)

    random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        param_distributions=rf_param_grid,
        cv=cv_folds,
        n_iter=n_iter,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X_train, y_train)
    best_params = random_search.best_params_

    final_model = random_search.best_estimator_

    y_test_pred      = final_model.predict(X_test)
    y_test_pred_prob = final_model.predict_proba(X_test)[:, 1]

    metrics = {
        "best_params": best_params,
        "test_metrics": {
            "accuracy": accuracy_score(y_test, y_test_pred),
            "f1":       f1_score(y_test, y_test_pred),
            "auroc":    roc_auc_score(y_test, y_test_pred_prob),
        },
        "model": final_model
    }

    return metrics

Endpoint 1: Caco-2 (Wang)

In [13]:
train, test = load_split_pickle('Caco2_Wang')

X_train, y_train, X_test, y_test = load_embeddings('Caco2_Wang', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Caco2_Wang', "metrics_MoLFormer_embeddings.txt", task="regression")

FileNotFoundError: [Errno 2] No such file or directory: 'Caco2_Wang_split.pkl'

Endpoint2: Lipophilicity

In [ ]:
train, test = load_split_pickle('Lipophilicity_AstraZeneca')

X_train, y_train, X_test, y_test = load_embeddings('Lipophilicity_AstraZeneca', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Lipophilicity_AstraZeneca', "metrics_MoLFormer_embeddings.txt", task="regression")

Endpoint 3: Solubility

In [ ]:
train, test = load_split_pickle('Solubility_AqSolDB')

X_train, y_train, X_test, y_test = load_embeddings('Solubility_AqSolDB', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Solubility_AqSolDB', "metrics_MoLFormer_embeddings.txt", task="regression")

Endpoint 4: HIA

In [ ]:
train, test = load_split_pickle('HIA_Hou')

X_train, y_train, X_test, y_test = load_embeddings('HIA_Hou', train, test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'HIA_Hou', "metrics_MoLFormer_embeddings.txt")

Endpoint 5: Half Life

In [ ]:
train, test = load_split_pickle('Half_Life_Obach')

X_train, y_train, X_test, y_test = load_embeddings('Half_Life_Obach', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Half_Life_Obach', "metrics_MoLFormer_embeddings.txt", task="regression")

Endpoint 6: Clearance Hepatocyte

In [ ]:
train, test = load_split_pickle('Clearance_Hepatocyte_AZ')

X_train, y_train, X_test, y_test = load_embeddings('Clearance_Hepatocyte_AZ', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'Clearance_Hepatocyte_AZ', "metrics_MoLFormer_embeddings.txt", task="regression")

Endpoint 7: CYP3A4 Inhibition

In [ ]:
train, test = load_split_pickle('CYP3A4_Veith')

X_train, y_train, X_test, y_test = load_embeddings('CYP3A4_Veith', train, test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'CYP3A4_Veith', "metrics_MoLFormer_embeddings.txt")

Endpoint 8: VDss

In [ ]:
train, test = load_split_pickle('VDss_Lombardo')

X_train, y_train, X_test, y_test = load_embeddings('VDss_Lombardo', train, test)

metrics = train_rf_regression(X_train, X_test, y_train, y_test)

print_metrics(metrics, task="regression")
save_metrics(metrics, 'VDss_Lombardo', "metrics_MoLFormer_embeddings.txt", task="regression")

Endpoint 9: AMES Mutagenicity

In [ ]:
train, test = load_split_pickle('AMES')

X_train, y_train, X_test, y_test = load_embeddings('AMES', train, test)

metrics = train_rf_classification(X_train, X_test, y_train, y_test)

print_metrics(metrics)
save_metrics(metrics, 'AMES', "metrics_MoLFormer_embeddings.txt")